<!--- sf-header --->
<table align="left">
<tr>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fstatmike%2Fscale-forecasting%2Fmain%2Fnotebooks%2F01_spark_via_connect.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Colab Enterprise logo">
      <br>Run in<br>Colab Enterprise
    </a>
  </td>
</tr>
</table>
<br clear="left"/>

> **Run in Colab Enterprise:** click the badge to import this notebook, pick a runtime, and
> **Run all**. The Terraform-deployed templates already carry the `SF_*` run identity in their env,
> so there's no environment cell to fill in. Runs on the **`sf-main`** runtime template (Python 3.11). See
> [`docs/notebook_runtimes.md`](https://github.com/statmike/scale-forecasting/blob/main/docs/notebook_runtimes.md)
> for the per-notebook template mapping and the headless acceptance harness.


# 01 · Spark models via Spark Connect

Run the **Spark UDF fan-out** — `groupBy(bucket).applyInPandas`, one task per `(series, model)` cell — on a real **Dataproc cluster**, driven from this notebook over a **Spark Connect** endpoint. The *same* engine code runs here as in a production Dataproc batch (the injectable-session seam, G1): the notebook just hands the engine a caller-owned session instead of letting it self-create one.

> **Reachability.** Spark Connect needs outbound access to the Dataproc endpoint. The first cell after the session does a scratch `spark.range(5).count()`; if it can't reach the endpoint, use the **remote-batch fallback** at the bottom — identical engine, submitted as a Dataproc batch.

## Get the code (cloud runtimes only)

On a cloud notebook (Colab Enterprise, Vertex Workbench) this clones or updates the repo so you're on the latest `src/`. **Skip it in a local clone** — it's a no-op guarded on the package already being importable.

In [ ]:
# Cloud bootstrap: clone + editable-install so `import scale_forecasting` resolves.
# Harmless locally — if the package already imports, we do nothing.
# We install the [spark] extra because this notebook drives the engine over a Spark Connect
# session, which needs the client-only `dataproc-spark-connect` (never imported by the engine).
import importlib.util
import os
import subprocess
import sys

REPO_URL = os.environ.get("SF_REPO_URL", "https://github.com/statmike/scale-forecasting.git")
REPO_DIR = os.environ.get("SF_REPO_DIR", "scale-forecasting")

if importlib.util.find_spec("scale_forecasting") is None:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[spark]"], check=True)
    sys.path.insert(0, os.path.join(REPO_DIR, "src"))

## Resolve the deployment (live GCP)

`Settings.resolve()` reads the `SF_*` environment — the *same* identity every writer uses (G1), so a notebook run and a Composer run land in the same registry. Required: `SF_PROJECT_ID`, `SF_CONNECTION`, `SF_WAREHOUSE_URI`; `SF_DATASET_ID`/`SF_REGION` default. This demo targets the live `run_registry` + `v_model_leaderboard` / `v_run_summary`.

In [ ]:
from google.cloud import bigquery

from scale_forecasting.settings import Settings

settings = Settings.resolve()
client = bigquery.Client(project=settings.project_id)
DATASET = settings.dataset_ref
print("deployment:", DATASET, "region:", settings.region)

## Review helpers

Registry rows written through the Storage Write API are *async-visible*, so we poll the leaderboard briefly until a run's models show up. `leaderboard(run_id)` returns one row per model — `compute_engine` splits Spark / BigQuery / ensemble — and `run_summary(run_id)` is the header roll-up.

In [ ]:
import time

import pandas as pd


def _query(sql, run_id):
    job = client.query(
        sql,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[bigquery.ScalarQueryParameter("run_id", "STRING", run_id)]
        ),
    )
    return job.result().to_dataframe()


def leaderboard(run_id, expect_models=None, tries=12, pause=3.0):
    """Poll v_model_leaderboard until `expect_models` all appear (or give up), newest metrics."""
    sql = (
        f"SELECT model_type, compute_engine, n_cells, mean_wape, mean_mae "
        f"FROM `{DATASET}.v_model_leaderboard` WHERE run_id=@run_id "
        f"ORDER BY mean_wape"
    )
    df = pd.DataFrame()
    for _ in range(tries):
        df = _query(sql, run_id)
        if expect_models is None or set(df["model_type"]) >= set(expect_models):
            break
        time.sleep(pause)
    return df


def run_summary(run_id):
    sql = f"SELECT * FROM `{DATASET}.v_run_summary` WHERE run_id=@run_id"
    return _query(sql, run_id)

## Stand up a Spark Connect session

A `DataprocSparkSession` pinned to **runtime 2.3** — the Spark Connect floor, whose workers run **Python 3.11**, matching the `sf-main` kernel and the project container (so driver↔worker Python parity holds with no special kernel). We attach the project's **container image** (`SF_CONTAINER_IMAGE`) so the workers carry the third-party deps the engine imports (`holidays`, `statsmodels`, `xgboost`, …); the `addArtifacts` call ships our own **source** — image = deps, artifacts = source, the same split the Dataproc batch uses. `dataproc-spark-connect` is the `[spark]` extra — a **client-only** dep, never imported by the engine (which runs on-cluster). Set `SF_DATAPROC_REGION` / `SF_DATAPROC_SUBNET` for your deployment. `SF_COMPUTE_SA` (baked into the template) is the runtime SA the session runs as — it carries the `dataproc.worker` permissions the session needs; the runner impersonates it.

In [ ]:
from google.cloud.dataproc_spark_connect import DataprocSparkSession
from google.cloud.dataproc_v1 import Session

region = os.environ.get("SF_DATAPROC_REGION", settings.region)
subnet = os.environ.get("SF_DATAPROC_SUBNET")  # e.g. projects/<p>/regions/<r>/subnetworks/<s>
compute_sa = os.environ.get("SF_COMPUTE_SA")  # runtime SA the session runs as (has dataproc.worker)
container_image = os.environ.get("SF_CONTAINER_IMAGE")  # the project's deps image (holidays, etc.)

session_cfg = Session()
# Runtime 2.3 is the Spark Connect floor, and its workers run Python 3.11 — matching sf-main and the
# project container, so the driver↔worker Python parity applyInPandas requires holds automatically.
session_cfg.runtime_config.version = "2.3"
# Attach the project's container image: it carries the third-party deps the engine imports on the
# workers (holidays, statsmodels, xgboost, …) that the stock runtime image lacks. Without it the
# shared pre-fit path fails (ModuleNotFoundError: holidays) and every cell errors silently. The
# addArtifacts call below still ships the scale_forecasting SOURCE — image = deps, artifacts = source.
if container_image:
    session_cfg.runtime_config.container_image = container_image
if subnet:
    session_cfg.environment_config.execution_config.subnetwork_uri = subnet
# Run the session runtime AS the compute SA — same identity the batch path uses. The session runtime
# needs dataprocrm.nodes.mintOAuthToken (a roles/dataproc.worker permission the compute SA carries,
# not the runner). The caller (runner) holds serviceAccountUser on compute, so it can set this.
if compute_sa:
    session_cfg.environment_config.execution_config.service_account = compute_sa

spark = (
    DataprocSparkSession.builder.projectId(settings.project_id)
    .location(region)
    .dataprocSessionConfig(session_cfg)
    .getOrCreate()
)
print("Spark Connect session up:", spark.version)

# Ship the scale_forecasting package SOURCE to the session's WORKERS. The explode fan-out pickles the
# group-runner closure on this client and runs it on the executors via applyInPandas; without the
# package on the worker sys.path the UDF deserialization fails ModuleNotFoundError. This adds the
# SAME zip the Dataproc batch delivers via python_file_uris (built by code_delivery), so the
# interactive-Connect and batch workers run byte-identical code (G1). The container image (above)
# supplies the third-party deps; this zip supplies our own source. addArtifacts needs a LOCAL file,
# so we write the zip to a temp path first; pyfile=True inserts it onto the executor path.
from scale_forecasting.code_delivery import write_package_zip  # noqa: E402

pkg_zip = write_package_zip()
spark.addArtifacts(str(pkg_zip), pyfile=True)
print("shipped package to workers:", pkg_zip.name)

## Reachability check

A trivial job proves the endpoint is reachable before we launch real work. If this raises, skip to the fallback.

> **Driver ↔ worker Python parity.** The explode fan-out ships Python to the workers via `applyInPandas`, and Spark Connect refuses to run mismatched Python minors. **Runtime 2.3 workers run Python 3.11**, which matches the `sf-main` kernel this notebook runs on — so parity holds automatically, with no special kernel to select. If you drive this from a different Python minor, use the **remote-batch fallback** at the bottom (it runs the identical engine on-cluster with no local driver, so parity is guaranteed).

In [ ]:
assert spark.range(5).count() == 5
print("endpoint reachable — Spark Connect is live")

## Parameters — edit me

Everything that shapes the run is set here as plain Python, then assembled into a **`RunConfig`** — the one frozen object that drives the run and is logged verbatim to `run_registry.raw_config`, so *this cell is the experiment record* (G2/G3). No separate JSON file to open.

`spark_explode.run(cfg, spark=session)` then runs the cross-join → bucket → `applyInPandas` fan-out **against the injected session**. Because we pass `spark`, the engine uses it and does **not** stop it (the caller owns its lifecycle). `manage_header=True` (the default) means this standalone run owns its own registry header.

- **`RUN_NAME`** carries a timestamp so each execution is its own clean run (the cell tables are append-only, so a fresh `run_id` avoids overwriting a still-buffering prior run).
- **`SOURCE_TABLE`** picks the storage format of the shipped input. The example ships in **both** `source_series_iceberg` (managed Apache Iceberg) and `source_series_native` (native BigQuery) — identical series, so you can flip the suffix to benchmark Spark reading either format. The engine reads both transparently via the spark-bigquery connector.
- **`MODELS`** all run as Spark cells (`compute_engine='spark'`). **`SPARK_METHOD='explode'`** is the per-cell fan-out (the hero path).
- **`SERIES_LIMIT`** subsets the seed so the demo is quick; raise it (or set `None`) to scale on the same series.

In [ ]:
from scale_forecasting.config import RunConfig
from scale_forecasting.engines import spark_explode
from scale_forecasting.registry.ids import make_run_id

# === Parameters — edit me ===============================================
RUN_NAME = f"nb01 spark connect {int(time.time())}"  # timestamped → a fresh run each execution
SOURCE_TABLE = "source_series_iceberg"  # shipped seed (100k); _iceberg↔_native to compare storage
MODELS = ["theta", "holtwinters", "sarimax", "xgboost"]
SPARK_METHOD = "explode"  # per-(series, model) cell fan-out (the hero path)
HORIZON = 28
SERIES_LIMIT = 100  # the demo scale (same first 100 series every approach uses)
HOLIDAYS = ["US"]
TRANSFORM = "log1p"  # none | log1p | boxcox
BACKTEST = True  # OOF metric panel so the leaderboard is scored (mean_wape / mean_mae)
N_FOLDS = 2
# ========================================================================

cfg = RunConfig(
    run_name=RUN_NAME,
    python_runtime="spark",
    spark_method=SPARK_METHOD,
    data={"source_table": SOURCE_TABLE, "horizon": HORIZON, "series_limit": SERIES_LIMIT},
    models=MODELS,
    features={"holidays": HOLIDAYS, "transform": TRANSFORM},
    backtest={"enabled": BACKTEST, "n_folds": N_FOLDS, "horizon": HORIZON, "step": HORIZON},
    compute={"persist_models": True},
)
run_id = make_run_id(cfg)
print("run_id:", run_id, "| models:", cfg.models, "| method:", cfg.spark_method)

spark_explode.run(cfg, settings=settings, spark=spark)
print("explode run complete:", run_id)

## Review — Spark cells on the leaderboard

Every model ran as Spark cells (`compute_engine='spark'`) under one `run_id`.

In [ ]:
board = leaderboard(run_id, expect_models=cfg.models)
board

In [ ]:
run_summary(run_id)

## Fallback — same engine as a remote Dataproc batch

If the Connect endpoint isn't reachable from this environment, `main.run(cfg)` with **no** injected session submits the identical engine as a remote Dataproc Serverless batch. Same `run_id`, same leaderboard — the injectable-session seam means there's no second code path to trust.

This is an **escape hatch**, not part of the Run-all flow (it would submit a second, redundant run), so it's shown here as a snippet — copy it into a new cell and run it only if the reachability check above failed:

```python
from scale_forecasting import main

# No `spark=` → remote Dataproc batch (the proven production path). Same cfg → same run_id.
fallback_run_id = main.run(cfg)
leaderboard(fallback_run_id, expect_models=cfg.models)
```